# Data Preparation for LLM Fine-tuning

This notebook covers the essential steps for preparing data before fine-tuning large language models. You'll learn:

1. **Tokenization**: Converting text into numerical tokens that models can process
2. **Padding and Truncation**: Ensuring all sequences have consistent lengths
3. **Instruction Dataset Formatting**: Structuring data for instruction-following tasks
4. **Batch Processing**: Efficiently tokenizing entire datasets
5. **Train/Test Splits**: Creating validation sets for model evaluation

Understanding data preparation is crucial because the quality and format of your training data directly impacts model performance.

In [ ]:
# Import necessary libraries for data preparation
import pandas as pd  # For working with tabular data
import datasets  # Hugging Face datasets library for efficient dataset handling

from pprint import pprint  # Pretty print for better output formatting
from transformers import AutoTokenizer  # Tokenizer for converting text to tokens

In [ ]:
from dotenv import load_dotenv
load_dotenv()


### Tokenizing Text

**Tokenization** is the process of converting human-readable text into numerical tokens that machine learning models can understand. 

- Each token typically represents a word or subword unit
- The tokenizer maps text to a vocabulary of tokens learned during pre-training
- Different models use different tokenizers (e.g., BPE, SentencePiece)

Let's start by loading a tokenizer and seeing how it converts text into tokens.

In [ ]:
# Load a pre-trained tokenizer from Hugging Face
# Pythia-70m is a small model suitable for learning and experimentation
# The tokenizer contains the vocabulary and encoding rules learned during pre-training
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-70m")

In [ ]:
# Example text to tokenize
# This is a simple sentence we'll convert into numerical tokens
text = "Hi, how are you?"

In [ ]:
print(tokenizer(text))

In [ ]:
# Tokenize the text and extract the input_ids
# input_ids are the numerical representation of tokens
# The tokenizer returns a dictionary with various fields; input_ids contains the token IDs
encoded_text = tokenizer(text)["input_ids"]

In [ ]:
# Display the encoded tokens
# Each number represents a token ID from the model's vocabulary
# Notice how the text is broken down into subword units
print(encoded_text)

In [ ]:
# Decode the tokens back to human-readable text
# This demonstrates that tokenization is reversible (lossless for most cases)
# The decode function converts token IDs back to the original text format
decoded_text = tokenizer.decode(encoded_text)
print("Decoded tokens back into text: ", decoded_text)

### Tokenizing Multiple Texts at Once

When working with datasets, you'll often need to tokenize multiple texts simultaneously. The tokenizer can efficiently handle batches of texts, which is much faster than processing them one by one.

In [ ]:
# Tokenize multiple texts in a single call
# Pass a list of strings to the tokenizer for batch processing
# Notice that each text produces a different number of tokens
list_texts = ["Hi, how are you?", "I'm good", "Yes"]
encoded_texts = tokenizer(list_texts)
print("Encoded several texts: ", encoded_texts["input_ids"])

### Padding and Truncation

Neural networks require inputs of fixed dimensions, but texts have variable lengths. We solve this with:

- **Padding**: Adding special tokens to shorter sequences to match the longest sequence in a batch
- **Truncation**: Cutting longer sequences to fit within a maximum length limit

These techniques ensure all sequences have the same length for efficient batch processing.

In [ ]:
# Set the padding token (using EOS token as padding token is common)
# Padding ensures all sequences in a batch have the same length
# padding=True pads sequences to match the longest sequence in the batch
tokenizer.pad_token = tokenizer.eos_token 
encoded_texts_longest = tokenizer(list_texts, padding=True)
print("Using padding: ", encoded_texts_longest["input_ids"])

In [ ]:
# Truncation cuts sequences that exceed max_length
# By default, truncation removes tokens from the right (end of sequence)
# max_length=3 means each sequence will have at most 3 tokens
encoded_texts_truncation = tokenizer(list_texts, max_length=3, truncation=True)
print("Using truncation: ", encoded_texts_truncation["input_ids"])

In [ ]:
# Change truncation to remove tokens from the left (beginning of sequence)
# Left-side truncation is useful when you want to keep the end of the text
# This is common in tasks where recent context is more important
tokenizer.truncation_side = "left"
encoded_texts_truncation_left = tokenizer(list_texts, max_length=3, truncation=True)
print("Using left-side truncation: ", encoded_texts_truncation_left["input_ids"])

In [ ]:
# Combine both padding and truncation
# This ensures all sequences have exactly max_length tokens
# Shorter sequences are padded, longer sequences are truncated
encoded_texts_both = tokenizer(list_texts, max_length=3, truncation=True, padding=True)
print("Using both padding and truncation: ", encoded_texts_both["input_ids"])

### Preparing Instruction Datasets

For instruction-following fine-tuning, we need to structure our data in a specific format:
- **Question/Prompt**: The input instruction or question
- **Answer/Response**: The desired output

We'll also create a prompt template that formats questions consistently. This helps the model learn the expected input-output pattern.

In [ ]:
import pandas as pd

# Load the instruction dataset from a JSONL file (JSON Lines format)
# JSONL is a common format where each line is a separate JSON object
filename = "lamini_docs.jsonl"
instruction_dataset_df = pd.read_json(filename, lines=True)
examples = instruction_dataset_df.to_dict()
examples

In [ ]:
for key,values in examples.items():
  print(key)
  print(values)
  if key == "question":
    print("First Question:", values[0])
  if key == "answer":
    print("First Answer:", values[0])
  print("--------------------------------")

In [ ]:
# Handle different dataset formats that might use different field names
# This code checks for common field name variations to make the code more flexible
# Note: The 'text' variable created here is for inspection/debugging purposes only
# It's not used in the rest of the code, but can be helpful to see what the data looks like
if "question" in examples and "answer" in examples:
  text = examples["question"][0] + examples["answer"][0]
elif "instruction" in examples and "response" in examples:
  text = examples["instruction"][0] + examples["response"][0]
elif "input" in examples and "output" in examples:
  text = examples["input"][0] + examples["output"][0]
else:
  text = examples["text"][0]

In [ ]:
print("First example:", text)

In [ ]:
len(examples["question"])

In [ ]:

# Define a prompt template that formats questions consistently
# This template helps the model understand the structure of inputs
# The {question} placeholder will be filled with actual questions
prompt_template = """### Question:
{question}

### Answer:"""

# Process all examples in the dataset
# Create a list of dictionaries, each containing formatted question and answer
num_examples = len(examples["question"])
finetuning_dataset = []
for i in range(num_examples):
  question = examples["question"][i]
  answer = examples["answer"][i]
  # Format the question using the template
  text_with_prompt_template = prompt_template.format(question=question)
  # Store both the formatted question and the answer
  finetuning_dataset.append({"question": text_with_prompt_template, "answer": answer})

from pprint import pprint
print("One datapoint in the finetuning dataset:")
pprint(finetuning_dataset[0])

### Tokenizing a Single Example

Before tokenizing the entire dataset, let's practice on a single example. This helps us understand:
- How to combine question and answer for tokenization
- How to handle sequence length limits
- What the tokenized output looks like

In [ ]:
print(finetuning_dataset[0]["question"])

In [ ]:
# Combine question and answer into a single text string
# For fine-tuning, we typically tokenize the full input-output pair together
text = finetuning_dataset[0]["question"] + finetuning_dataset[0]["answer"]

print(text)

In [ ]:
# Tokenize with padding
# return_tensors="np" returns NumPy arrays (useful for some frameworks)
# padding=True ensures consistent length (though with single example, no padding needed)
tokenized_inputs = tokenizer(
    text,
    return_tensors="np",
    padding=True,
    max_length=100
)
pprint(tokenized_inputs["input_ids"])

In [ ]:
print(tokenized_inputs["input_ids"].shape)

In [ ]:
pprint(tokenized_inputs)

In [ ]:
# Determine the maximum sequence length to use
# Set a desired max_length (2048 tokens is common for many models)
# But use the actual sequence length if it's shorter (no need to pad unnecessarily)
max_length = 2048
max_length = min(
    tokenized_inputs["input_ids"].shape[1],  # Actual length of this sequence
    max_length,  # Desired maximum length
)

In [ ]:
# Re-tokenize with truncation applied
# truncation=True ensures sequences longer than max_length are cut
# This is important for managing memory and computational costs
tokenized_inputs = tokenizer(
    text,
    return_tensors="np",
    truncation=True,
    max_length=max_length
)

In [ ]:
# Display the final tokenized input IDs
# These are the numerical tokens ready for model training
pprint(tokenized_inputs["input_ids"])

### Tokenizing the Entire Instruction Dataset

Now we'll create a function to tokenize the entire dataset efficiently. The `datasets` library's `map` function allows us to apply tokenization to all examples in parallel, which is much faster than processing them sequentially.

**How it works:**
- The `tokenize_function` receives a **batch** of examples (not just one)
- The `map()` function calls `tokenize_function` **multiple times** - once for each batch
- Each batch contains multiple examples (controlled by `batch_size`)
- The function processes ALL examples in each batch, then `map()` moves to the next batch
- This continues until **all examples** in the dataset have been processed

**Example:** If you have 1000 examples and `batch_size=10`, the function is called 100 times, each time processing 10 examples.

In [ ]:
def tokenize_function(examples):
    """
    Tokenize examples from the dataset.
    
    This function handles different dataset formats and applies tokenization
    with appropriate padding and truncation settings.
    
    Args:
        examples: A batch of examples from the dataset
        
    Returns:
        Tokenized inputs with input_ids ready for model training
    """
    # Handle different field name formats in the dataset
    # Extract the text to tokenize based on available fields
    if "question" in examples and "answer" in examples:
      text = examples["question"][0] + examples["answer"][0]
    elif "input" in examples and "output" in examples:
      text = examples["input"][0] + examples["output"][0]
    else:
      text = examples["text"][0]

    # Set padding token (using EOS token is common practice)
    tokenizer.pad_token = tokenizer.eos_token
    
    # First pass: tokenize with padding to see actual length
    # This helps determine if truncation is needed
    tokenized_inputs = tokenizer(
        text,
        return_tensors="np",
        padding=True,
    )

    # Calculate the maximum length to use
    # Use the smaller of: actual sequence length or desired max (2048)
    max_length = min(
        tokenized_inputs["input_ids"].shape[1],
        2048
    )
    
    # Set truncation to left side (keeps the end of sequences)
    # This is useful for keeping the answer/response intact
    tokenizer.truncation_side = "left"
    
    # Re-tokenize with truncation applied
    # This ensures sequences fit within the model's context window
    tokenized_inputs = tokenizer(
        text,
        return_tensors="np",
        truncation=True,
        max_length=max_length
    )

    return tokenized_inputs

In [ ]:
# Load the dataset using Hugging Face datasets library
# This provides efficient data loading and processing capabilities
finetuning_dataset_loaded = datasets.load_dataset("json", data_files=filename, split="train")

# Display the loaded dataset (features and row count)
finetuning_dataset_loaded

In [ ]:
print(finetuning_dataset_loaded["question"][0])
print(finetuning_dataset_loaded["answer"][0])
print(len(finetuning_dataset_loaded["question"][0]) + len(finetuning_dataset_loaded["answer"][0]))

In [ ]:
tokenized_dataset = finetuning_dataset_loaded.map(
    tokenize_function,
    batched=True,
    batch_size=1,
    drop_last_batch=True
)

print(tokenized_dataset)

In [ ]:
# Number of examples in the tokenized dataset
print(len(tokenized_dataset["input_ids"]))
print(tokenized_dataset["question"][0])
print(tokenized_dataset["answer"][0])
# Character count (question + answer) — measured in *characters*
print(len(tokenized_dataset["question"][0]) + len(tokenized_dataset["answer"][0]))
# Token count for same example — measured in *tokens* (one token ≈ multiple characters, so 358 chars → 77 tokens is normal)
print(len((tokenized_dataset["input_ids"][0])))  # first example
print(len((tokenized_dataset["input_ids"][1])))  # second example

In [ ]:
# Add labels column for training
# In language modeling, labels are typically the same as input_ids
# The model learns to predict the next token, so labels = input_ids shifted by one position
# For simplicity, we use input_ids as labels (the training loop handles shifting)
tokenized_dataset = tokenized_dataset.add_column("labels", tokenized_dataset["input_ids"])

In [ ]:
print(tokenized_dataset)

### Preparing Train/Test Splits

It's essential to split your data into training and validation sets:
- **Training set**: Used to teach the model
- **Test/Validation set**: Used to evaluate model performance and prevent overfitting

A common split is 90% training and 10% validation. The split should be shuffled randomly to ensure both sets are representative of the overall data distribution.

In [ ]:
# Split the dataset into training and test sets
# test_size=0.1 means 10% of data goes to test set, 90% to training
# shuffle=True randomly shuffles data before splitting (important for unbiased splits)
# seed=123 sets random seed for reproducibility (same split every time)
split_dataset = tokenized_dataset.train_test_split(test_size=0.1, shuffle=True, seed=123)
print(split_dataset)

### Example Datasets to Try

The Hugging Face Hub hosts many pre-processed datasets ready for fine-tuning. Here are some examples you can experiment with. These datasets are already formatted for instruction-following tasks and can be loaded directly.

In [ ]:
# Load a dataset directly from Hugging Face Hub
# This dataset is already formatted for instruction-following tasks
# You can browse more datasets at https://huggingface.co/datasets
finetuning_dataset_path = "lamini/lamini_docs"
finetuning_dataset = datasets.load_dataset(finetuning_dataset_path)
print(finetuning_dataset)

In [ ]:
# Additional example datasets you can experiment with
# These are pre-processed datasets available on Hugging Face Hub
# Each contains instruction-response pairs for fine-tuning
taylor_swift_dataset = "lamini/taylor_swift"  # Dataset about Taylor Swift
bts_dataset = "lamini/bts"  # Dataset about BTS (K-pop group)
open_llms = "lamini/open_llms"  # General open-source LLM dataset

In [ ]:
# Load and inspect one of the example datasets
# This shows what a typical instruction dataset looks like
# Notice the structure: each example has fields like 'instruction' and 'output'
dataset_swiftie = datasets.load_dataset(taylor_swift_dataset)
print(dataset_swiftie["train"][1])

In [ ]:
# Optional: Push your prepared dataset to Hugging Face Hub
# This allows you to share datasets or use them across different projects
# 
# Steps to push a dataset:
# 1. Install huggingface_hub: !pip install huggingface_hub
# 2. Login to Hugging Face: !huggingface-cli login
# 3. Push your dataset: split_dataset.push_to_hub(dataset_path_hf)
#
# Note: Replace dataset_path_hf with your desired dataset name (e.g., "username/dataset-name")
# This is how to push your own dataset to your Huggingface hub
# !pip install huggingface_hub
# !huggingface-cli login
# split_dataset.push_to_hub(dataset_path_hf)